In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

import math
from dataclasses import dataclass
from torchvision.datasets import CIFAR10
from torchvision import transforms

from torch.utils.data import DataLoader, random_split



class MultiHeadAttention(nn.Module):
    """Models a mutli-head self-attention block for an Encoder"""

    def __init__(self, n_heads: int, embd_size: int) -> None:
        """Initializes the mult-head self-attention block.
        
        Args:
            n_heads: Number of attention heads in the attention head of each encoder.
            embd_size: Dimension size of the patch embeddings.
        """
        super().__init__()

        self.n_heads = n_heads
        self.embd_size = embd_size

        assert self.embd_size % self.n_heads == 0
        self.head_embd_size = self.embd_size // self.n_heads

        self.QKV = nn.Linear(self.embd_size, self.n_heads * 3*self.head_embd_size)

        # Linearly combines the output from each attention head
        self.W_O = nn.Linear(self.embd_size, self.embd_size)

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        """Computes mutli-head self-attention over the input token sequence.
        
        Args:
            x: A tensor of shape (T, embd_size), where T is the number of total number of tokens.

        Returns:
            A tensor of shape(T, embd_size) containing the contextualized representation of each input token.
        """
        T = x.shape[0]

        qkv = self.QKV(x).reshape(T, self.n_heads, -1).transpose(1, 0) # (n_heads, T, 3*head_embd_size)
        
        Q, K, V = qkv.split(self.head_embd_size, dim = -1) # (n_heads, T, head_embd_size)

        # (n_heads, T, T)
        attn_scores = torch.softmax((Q @ K.transpose(2, 1)) / math.sqrt(self.head_embd_size), dim = -1)
        z = attn_scores @ V # (n_heads, T, head_embd_size)
        z = z.transpose(1, 0).reshape(T, -1) # (T, n_heads * head_embd_size) = (T, embd_size)
        z = self.W_O(z) # (T, embd_size)

        return z


class MLP(nn.Module):
    """Models a simple Multi-Layer Perception"""

    """Initializes a MLP
    Args:
        hidden_size: Number of hidden neurons
        output_size: Number of output neurons (if applicable)
    """
    def __init__(self, embd_size: int, hidden_size: int, output_size: int = None):
        super().__init__()
        self.layer1 = nn.Linear(embd_size, hidden_size)
        self.gelu = nn.GELU()

        if output_size is not None:
            self.layer2 = nn.Linear(hidden_size, output_size)
        else: 
            self.layer2 = nn.Linear(hidden_size, embd_size)

    def __call__(self, x):
        return self.forward(x)
    """Maps the input tensor onto hidden_size neurons, and then linearly maps those neurons back onto
    the embedding dimension. However, if an output_size was given, then those hidden neurons will be 
    mapped onto the output dimension.

    Args:
        x: A tensor of shape (T, embd_size), where T is the number of total number of tokens.
    """
    def forward(self, x):
        return self.layer2(self.gelu(self.layer1(x)))


class Classifier(nn.Module):
    def __init__(self, embd_size, hidden_size, output_size):
        super().__init__()
        self.classifer = MLP(embd_size, hidden_size, output_size)
        self.classifier_fine_tune = nn.Linear(embd_size, output_size)

    def __call__(self, x, fine_tune):
        return self.forward(x, fine_tune)

    def forward(self, x, fine_tune):
        if fine_tune: 
            return self.classifier_fine_tune(x)
        return self.classifer(x)
    

class EncoderBlock(nn.Module):
    def __init__(self, n_heads, embd_size):
        super().__init__()
        self.ln = nn.LayerNorm(embd_size)
        self.msa = MultiHeadAttention(n_heads, embd_size)
        self.mlp = MLP(embd_size, 4*embd_size)

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        return self.msa(self.ln(x)) + x

@dataclass
class ViTConfig:
    """Stores configuration parameters for a ViT.
    
    Atrributes:
        n_heads: Number of attention heads in the attention head of each encoder.
        embd_size: Dimension size of the patch embeddings.
        patch_size: Length of each image patch.
        config.: Number of encoders in the ViT.
        input_shape: Tuple containing the height, width, and channels of the input images.
        output_size: Number of image classes.
    """
    n_heads: int = 4
    embd_size: int = 128
    patch_size: int = 2
    num_encoders: int = 4
    input_shape: tuple = (32, 32, 3)
    output_size: int = 10


class ViT(nn.Module):
    def __init__(self, config: ViTConfig):
        super().__init__()
        self.config = config

        self.patch_size = config.patch_size
        H, W, C = config.input_shape

        assert H % config.patch_size == 0 and W % config.patch_size == 0, f"Invalid patch size {config.patch_size} for {H}x{W} image."
        num_patches = H*W // config.patch_size**2


        self.E = nn.Linear(config.patch_size**2 * C, config.embd_size, bias=False)
        self.E_pos = nn.Parameter(torch.randn(size = (num_patches+1, config.embd_size)))
        self.CLS = nn.Parameter(torch.randn(size = (config.embd_size,)))

        self.encoders = nn.ModuleList(
            config.num_encoders * [EncoderBlock(config.n_heads, config.embd_size)]
        )

        self.classifier = Classifier(config.embd_size, 4*config.embd_size, config.output_size)

    def __call__(self, x, fine_tune=False):
        return self.forward(x, fine_tune)

    def forward(self, x, fine_tune):
        H, W, C = x.shape


        image_patches = torch.stack([
            x[i:i+self.patch_size, j:j+self.patch_size, :].reshape(-1)
            for j in range(0, W, self.patch_size) 
            for i in range(0, H, self.patch_size)
        ]) # (num_patches, P*P*C)

        patch_embeddings = self.E(image_patches) # (num_patches, embd_size)
        patch_embeddings = torch.cat(
            [self.CLS.unsqueeze(0), patch_embeddings],
            dim = 0
        ) # (num_patches+1, embd_size)

        z = patch_embeddings + self.E_pos
        
        for encoder in self.encoders:
            z = encoder(z)

        return self.classifier(z[0], fine_tune)












In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x : x.permute(1, 2, 0)) # (C, H, W) -> (H, W, C)
])


train_dataset = CIFAR10(root="data", train=True, transform=transform, download=True)
test_dataset = CIFAR10(root="data", train=False, transform=transform, download=True)

torch.manual_seed(42)
train_dataset, val_dataset = random_split(train_dataset, [40000, 10000])

batch_size = 128

train_dataloader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    shuffle = False
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    shuffle = False
)

config = ViTConfig()
model = ViT(config)

lr = 1e-3
weight_decay = 1e-4

optimizer = torch.optim.AdamW(params=model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = torch.nn.CrossEntropyLoss()

total_loss = 0.0
model.train()
for i, train_batch in enumerate(train_dataloader):
    images, labels = train_batch

    optimizer.zero_grad()

    logits = model(images[0])
    loss = loss_fn(logits, labels[0])

    loss.backward()
    optimizer.step()    

    break
